## Imports

In [1]:
import sys
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.loaders.load_kaggle_data import load_data
from src.splits.splitting import create_stratified_kfold
from src.preprocessing.preprocessing import split_features_target
from src.preprocessing.preprocessing import get_feature_types
from src.preprocessing.preprocessing import (
    create_one_hot_preprocessor,
)
from src.feature_engineering.features import apply_features
from src.feature_engineering.features import FEATURES
from src.models.xgboost import build_xgboost
from src.submission.submission import create_submission

## Load data

In [2]:
train, test, sample_submission = load_data()

TARGET = "Will_Buy_EV"
ID_COLUMN = "id"

X, y = split_features_target(
    train,
    target=TARGET,
)

X = X.drop(columns=[ID_COLUMN])

X_test = test

print("X columns:")
print(X.columns.tolist())

print("\nX_test columns:")
print(X_test.columns.tolist())

print(f"\nNumber of features: {X.shape[1]}")
print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")

print("\nSame features in X and X_test:", X.columns.equals(X_test.columns))

X columns:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

X_test columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Number of features: 13
X shape: (668665, 13)
X_test shape: (286571, 14)

Same features in X and X_test: False


## Cross-validation strategy

In [3]:
cv = create_stratified_kfold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## Define best hyperparameters and build model

In [4]:
best_params = {
    "subsample": 1.0,
    "n_estimators": 1000,
    "max_depth": 3,
    "learning_rate": 0.1,
    "colsample_bytree": 0.9,
}

best_xgb = build_xgboost(
    **best_params
)

## Feature engineering

In [5]:
print("Available engineered features:")
for feature in FEATURES:
    print("-", feature)

Available engineered features:
- charging_total
- charging_difference
- charging_home_ratio
- charging_per_commute
- log_income
- income_per_car
- income_x_cars
- income_x_subsidy
- income_x_environmental
- medium_anxiety_indicator
- high_anxiety_indicator
- range_anxiety_score
- charging_x_medium_anxiety
- charging_x_high_anxiety
- commute_x_medium_anxiety
- commute_x_high_anxiety
- commute_x_home_charging
- commute_x_cars
- commute_per_car
- environmental_x_subsidy
- environmental_x_commute
- environmental_x_range_anxiety
- charging_x_home_charging
- environmental_x_charging


In [6]:
all_features = list(FEATURES.keys())

print(f"Number of engineered features: {len(all_features)}")

X_all_features = apply_features(
    X,
    all_features,
)

X_test_all_features = apply_features(
    X_test,
    all_features,
)

Number of engineered features: 24


## Numerical & Categorical features

In [7]:
numeric_features, categorical_features = get_feature_types(X_all_features)

preprocessor = create_one_hot_preprocessor(
    categorical_features=categorical_features,
)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))


Number of numeric features: 31
Number of categorical features: 6


## Preprocessor and pipeline

In [8]:
preprocessor = create_one_hot_preprocessor(
    categorical_features=categorical_features,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", best_xgb),
    ]
)

## Evaluate the model with all the features

In [9]:
scores = cross_val_score(
    model,
    X_all_features,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)

In [10]:
print("ROC-AUC scores:", scores)
print(f"Mean ROC-AUC: {scores.mean():.6f}")
print(f"Std ROC-AUC:  {scores.std():.6f}")

ROC-AUC scores: [0.94071016 0.94188065 0.94308055 0.94260632 0.94194085]
Mean ROC-AUC: 0.942044
Std ROC-AUC:  0.000801


In [11]:
model.fit(X_all_features, y)

# Get fitted components
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

# Feature names after preprocessing / one-hot encoding
feature_names = fitted_preprocessor.get_feature_names_out()

# XGBoost feature importances
importances = fitted_classifier.feature_importances_

feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance_df.head(30))

,feature,importance
0,remainder__Environmental_Concern_x_Subsidy,0.725636
1,categorical__Subsidy_Available_No,0.066468
2,remainder__Range_Anxiety_Score,0.061541
3,remainder__Environmental_Concern_Level,0.039752
4,categorical__Range_Anxiety_Level_Low,0.039160
5,remainder__Income_x_Environmental,0.031795
6,categorical__Home_Charging_Possible_No,0.008388
7,remainder__Income_x_Subsidy,0.004331
8,categorical__Range_Anxiety_Level_Medium,0.003804
9,remainder__Log_Income,0.002684


## Submission with all the features

In [ ]:
test_proba = model.predict_proba(X_test_all_features)[:, 1]

submission = create_submission(
    model=model,
    test=X_test_all_features,
    sample_submission=sample_submission,
    output_filename="xgb_all_features_2.csv",
    target=TARGET,
)

display(submission.head())